In [1]:
# Import live code changes in
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import geopandas as gpd
import rasterio
import pandas as pd
from functools import reduce

from sovereign.utils import (map_flopros_to_adm, union_gdfs, calculate_river_length_per_admin,
                            calculate_increased_protection, calculate_increased_protection_costs,
                            disaggregate_total_to_raster)

#### Set filepaths and provide data info

In [2]:
root = Path.cwd().parent # find project root
flopros_path = os.path.join(root, 'inputs', 'flood', 'protection', 'UGA_flopros.tif')
adm_path = os.path.join(root, 'inputs', 'boundaries', 'admin', 'gadm36_UGA_1.shp')
# mapping_csv = os.path.join(root, 'inputs', 'flood', 'protection', 'flopros-adm1-map.csv')
basin_path = os.path.join(root, 'inputs', 'boundaries', 'basins', 'BA_UGA_lev06.shp')
output_path = Path(os.path.join(root, 'outputs', 'boundaries', 'analysis_basins.gpkg'))
river_path = os.path.join(root, 'inputs', 'flood', 'rivers', 'hydroRIVERS_v10_UGA.shp')
urban_path = os.path.join(root, 'inputs', 'exposure', 'ghsl_du', 'ghsl_du_gadm_UGA.gpkg')

In [3]:
# Load data
# flopros = gpd.read_file(flopros_path)
adm = gpd.read_file(adm_path)
# mapping_df = pd.read_csv(mapping_csv)
basins = gpd.read_file(basin_path)
rivers = gpd.read_file(river_path)
urban = gpd.read_file(urban_path)

#### Baseline Flood Protection

In [4]:
# Map protection levels to the Admin Dataset
protection_levels = map_flopros_to_adm(adm, flopros_path)

In [5]:
# Merge protection levels with GADM dataset
# Rename the columns we want to merge
basins = basins.rename(columns={'HYBAS_ID': 'HYBAS_ID_06'})
protection_levels = protection_levels.rename(columns={'GID_1': 'flpr_gid_1', 'NAME_1': 'NAME'})
# Select relevant columns 
basins_selected = basins[['geometry', 'HYBAS_ID_06']]
protection_levels_selected = protection_levels[['geometry', 'flpr_gid_1', 'NAME', 'MerL_Riv']]
datasets = [basins_selected, protection_levels_selected]
merged = reduce(union_gdfs, datasets)

C:\Users\Mark.DESKTOP-UFHIN6T\anaconda3\envs\sovereign-risk\lib\site-packages\sovereign\utils.py:224: UserWarning: `keep_geom_type=True` in overlay resulted in 18 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return gpd.overlay(gdf1, gdf2_copy, how='union')


In [6]:
# Clean dataset (remove all rows where there are no column values consistently accross the rows)
columns_to_check = ["HYBAS_ID_06", "flpr_gid_1"]
# Create a boolean mask for non-null and non-zero values
mask = (merged[columns_to_check].notnull() & (merged[columns_to_check] != 0)).all(axis=1)
# Filter the GeoDataFrame to keep only rows that meet the criteria
filtered_merged = merged.loc[mask]

In [7]:
# Save to file
output_path.parent.mkdir(parents=True, exist_ok=True)
filtered_merged.to_file(output_path)

#### Prepare Exposure Datasets

In [8]:
# Inputs
AGR_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'UGA_cropland.tif')
INF_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'UGA_infra.tif')
RES_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'UGA_ghs-res_a.tif')
NRES_path = os.path.join(root, 'inputs', 'exposure', 'gridded', 'UGA_ghs-nres_a.tif')
capstock_path = os.path.join(root, 'inputs', 'exposure', 'capstock', 'giri_capstock.csv')
gva_path = os.path.join(root, 'inputs', 'exposure', 'gva', 'national_sectoral_gva.csv')
# Outputs
inf_pri_path = os.path.join(root, 'outputs', 'exposure', 'inf_priv_capstock.tif')
inf_pub_path = os.path.join(root, 'outputs', 'exposure', 'inf_pub_capstock.tif')
nres_pub_path = os.path.join(root, 'outputs', 'exposure', 'nres_pub_capstock.tif')
nres_pri_path = os.path.join(root, 'outputs', 'exposure', 'nres_priv_capstock.tif')
res_pri_path = os.path.join(root, 'outputs', 'exposure', 'res_priv_capstock.tif')
agr_gva_path = os.path.join(root, 'outputs', 'exposure', 'UGA_agr_GVA.tif')
man_gva_path = os.path.join(root, 'outputs', 'exposure', 'UGA_man_GVA.tif')
ser_gva_path = os.path.join(root, 'outputs', 'exposure', 'UGA_ser_GVA.tif')

In [9]:
# Load data
capstock = pd.read_csv(capstock_path)
gva = pd.read_csv(gva_path)

In [10]:
# Disaggregate exposure datasets
disaggregate_total_to_raster(gva['AGR'][0], AGR_path, agr_gva_path) # agriculture GVA
disaggregate_total_to_raster(gva['SER'][0], NRES_path, ser_gva_path) # service GVA
disaggregate_total_to_raster(gva['MAN'][0], NRES_path, man_gva_path) # manufacturing GVA
inf_pri_sum = capstock.loc[(capstock['variable_type']=='infrastructure') & (capstock['ownership_classification']=='Private'), "data_sum"].sum()
disaggregate_total_to_raster(inf_pri_sum, INF_path, inf_pri_path) # private infrastructure capital stock
inf_pub_sum = capstock.loc[(capstock['variable_type']=='infrastructure') & (capstock['ownership_classification']=='Public'), "data_sum"].sum()
disaggregate_total_to_raster(inf_pub_sum, INF_path, inf_pub_path) # private infrastructure capital stock
nres_pub_sum = capstock.loc[(capstock['variable_type']=='buildings') & (capstock['ownership_classification']=='Public'), "data_sum"].sum()
disaggregate_total_to_raster(nres_pub_sum, NRES_path, nres_pub_path) # non-residential public capital stock
res_pri_sum = capstock.loc[(capstock['sector_key']=='ic_low') | (capstock['sector_key']=='ic_mlow'), "data_sum"].sum()
disaggregate_total_to_raster(res_pri_sum, RES_path, res_pri_path) # residential private capital stock
nres_pri_sum = capstock.loc[(capstock['variable_type']=='buildings') & (capstock['ownership_classification']=='Private'), "data_sum"].sum() - res_pri_sum
disaggregate_total_to_raster(nres_pri_sum, NRES_path, nres_pri_path) # non-residential private capital stock

#### Adaptation Scenario

In [7]:
# User Config
min_urban = 23 # towns and cities see -> https://human-settlement.emergency.copernicus.eu/documents/GHSL_Data_Package_2023.pdf?t=1727170839#page=66.08
adaptation_target = 100 # return period
river_size_limit = 500 # upstream catchment area (km^2)
adaptation_unit_cost = 2399000 # $2.399 million per km unit cost from Boulange paper

In [10]:
# # Calculate river length within each basin
# adaptation = calculate_river_length_per_admin(protection_levels, rivers, river_size_limit, urban, min_urban)
# # Calculate how much additional protection is needed to reach a target protection level
# adaptation = calculate_increased_protection(adaptation, adaptation_target)
# # Calculate how much this additional protection will cost (using Boulange et al 2023 method)
# adaptation = calculate_increased_protection_costs(adaptation, adaptation_unit_cost)

In [14]:
# Create a raster mask with areas to be protected 
output_path = os.path.join(root, 'outputs', 'flood', 'adaptation', f'urban_mask_{min_urban}.tif')
ref_raster_path = os.path.join(root, 'inputs', 'flood', 'maps', 'UGA_jrc-flood_RP10.tif') # referance raster for creation
filtered_urbanisation = urban[urban['DEGURBA_L2'] >= min_urban]

# Open the reference raster to use its dimensions and CRS
with rasterio.open(ref_raster_path) as ref:
    # Create an empty mask with the same dimensions as the reference raster
    mask = rasterio.features.rasterize(
        ((geom, 1) for geom in filtered_urbanisation.geometry),
        out_shape=ref.shape,
        transform=ref.transform,
        fill=0,  # Fill value for 'background'
        all_touched=False,  # Only if centroids touch. 
        dtype='uint8'
    )

# Save the mask raster
with rasterio.open(
    output_path,
    'w',
    driver='GTiff',
    height=ref.height,
    width=ref.width,
    count=1,
    dtype='uint8',
    crs=ref.crs,
    transform=ref.transform,
) as dst:
    dst.write(mask, 1)

In [20]:
# print(f'Adapting Urban DUC{min_urban} and denser regions to RP{adaptation_target} costs:', adaptation['Add_Pr_c_u'].sum())